In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import os
os.chdir("c:/Users/matts/Desktop/wakeup-ai")
# ---------- 1. Load ----------
oxf = pd.read_csv(r"data\parkinson\oxford_12\processed_oxford_12_data.csv")
ist = pd.read_csv(r"data\parkinson\istanbul_updrs\processed_istanbul_updrs_data_scaled.csv")

# ---------- 2. Column mapping (Istanbul -> Oxford naming) ----------
rename_ist = {
    'jitter_local':     'Jitter(%)',
    'jitter_local_abs': 'Jitter(Abs)',
    'jitter_rap':       'Jitter:RAP',
    'jitter_ppq5':      'Jitter:PPQ5',
    'jitter_ddp':       'Jitter:DDP',
    'shimmer_local':    'Shimmer',
    'shimmer_local_db': 'Shimmer(dB)',
    'shimmer_apq3':     'Shimmer:APQ3',
    'shimmer_apq5':     'Shimmer:APQ5',
    'shimmer_apq11':    'Shimmer:APQ11',
    'shimmer_dda':      'Shimmer:DDA',
    'nth':              'NHR',   # Noise-to-Harmonics ↔ NHR
    'htn':              'HNR',   # Harmonics-to-Noise ↔ HNR
    'updrs':            'total_UPDRS',
    'class_label':      'status',
}
ist = ist.rename(columns=rename_ist)

OVERLAP = ['Jitter(%)','Jitter(Abs)','Jitter:RAP','Jitter:PPQ5','Jitter:DDP',
           'Shimmer','Shimmer(dB)','Shimmer:APQ3','Shimmer:APQ5','Shimmer:APQ11',
           'Shimmer:DDA','NHR','HNR']
OXFORD_ONLY = ['RPDE','DFA','PPE']             # to be imputed for Istanbul
ISTANBUL_ONLY = ['pitch_median','pitch_mean','pitch_std','pitch_min','pitch_max',
                 'num_pulses','mean_period','std_period',
                 'unvoiced_frames_frac','degree_voice_breaks','ac']

# ---------- 3. Severity bins (apply to Istanbul using same scheme as Oxford) ----------
def updrs_severity(x):
    if pd.isna(x):      return np.nan
    if x < 20:          return 'Mild (<20)'
    if x <= 35:         return 'Moderate (20-35)'
    return 'Severe (>35)'

ist['UPDRS_Severity'] = ist['total_UPDRS'].apply(updrs_severity)

# Oxford already has UPDRS_Severity. If not, regenerate:
if 'UPDRS_Severity' not in oxf.columns:
    oxf['UPDRS_Severity'] = oxf['total_UPDRS'].apply(updrs_severity)

# ---------- 4. Impute RPDE / DFA / PPE for Istanbul via regression on Oxford ----------
X_train = oxf[OVERLAP].values
imputed = {}
for target in OXFORD_ONLY:
    y_train = oxf[target].values
    model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    imputed[target] = model.predict(ist[OVERLAP].values)
    # Optional: report training R^2 so you know how trustworthy the imputation is
    print(f"{target} train R^2 = {model.score(X_train, y_train):.3f}")

for t, v in imputed.items():
    ist[t] = v
    ist[f'{t}_imputed'] = 1  # provenance flag
for t in OXFORD_ONLY:
    oxf[f'{t}_imputed'] = 0

# ---------- 5. Impute motor_UPDRS for Istanbul from total_UPDRS (linear) ----------
lin = LinearRegression().fit(oxf[['total_UPDRS']], oxf['motor_UPDRS'])
ist['motor_UPDRS'] = lin.predict(ist[['total_UPDRS']])
ist['motor_UPDRS_imputed'] = 1
oxf['motor_UPDRS_imputed'] = 0

# ---------- 6. Harmonize status column ----------
# Oxford: all PD -> status = 1
if 'status' not in oxf.columns:
    oxf['status'] = 1
# Istanbul already has status (0/1) from class_label

# ---------- 7. Add Istanbul-only features as NaN to Oxford, vice versa ----------
for c in ISTANBUL_ONLY:
    if c not in oxf.columns:
        oxf[c] = np.nan
for c in OXFORD_ONLY:
    if c not in ist.columns:
        ist[c] = np.nan

# ---------- 8. Source tagging ----------
oxf['source'] = 'oxford'
ist['source'] = ist.get('source', 'istanbul')

# ---------- 9. Align columns and concatenate ----------
common_cols = ['subject_id','status','source','total_UPDRS','motor_UPDRS',
               'UPDRS_Severity'] + OVERLAP + OXFORD_ONLY + ISTANBUL_ONLY + \
              ['RPDE_imputed','DFA_imputed','PPE_imputed','motor_UPDRS_imputed']

# Make subject_id unique across datasets
oxf['subject_id'] = 'OXF_' + oxf['subject_id'].astype(str)
ist['subject_id'] = 'IST_' + ist['subject_id'].astype(str)

combined = pd.concat([oxf[common_cols], ist[common_cols]], ignore_index=True)

# ---------- 10. (Recommended) Re-scale acoustic block jointly ----------
scaler = StandardScaler()
acoustic_all = OVERLAP + OXFORD_ONLY + ISTANBUL_ONLY
combined[acoustic_all] = scaler.fit_transform(
    combined[acoustic_all].fillna(combined[acoustic_all].mean())
)

combined.to_csv("combined_parkinson_voice.csv", index=False)
print(combined.shape, combined['source'].value_counts())

RPDE train R^2 = 0.948
DFA train R^2 = 0.949
PPE train R^2 = 0.978
(6915, 37) source
oxford            5875
istanbul_train    1040
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import cross_val_score, GroupKFold

# (a) Honest CV-R^2 on Oxford
gkf = GroupKFold(n_splits=5)
for target in ['RPDE', 'DFA', 'PPE']:
    scores = cross_val_score(
        RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
        oxf[OVERLAP], oxf[target],
        groups=oxf['subject_id'], cv=gkf, scoring='r2'
    )
    print(f"{target} CV R^2 = {scores.mean():.3f} ± {scores.std():.3f}")

# (b) Are imputed Istanbul values inside Oxford's range?
for t in ['RPDE', 'DFA', 'PPE']:
    o_lo, o_hi = oxf[t].quantile([0.01, 0.99])
    inside = ist[t].between(o_lo, o_hi).mean()
    print(f"{t}: {inside:.1%} of imputed Istanbul values inside Oxford's 1-99% range")


RPDE CV R^2 = 0.502 ± 0.059
DFA CV R^2 = 0.433 ± 0.134
PPE CV R^2 = 0.777 ± 0.048
RPDE: 87.3% of imputed Istanbul values inside Oxford's 1-99% range
DFA: 81.6% of imputed Istanbul values inside Oxford's 1-99% range
PPE: 66.8% of imputed Istanbul values inside Oxford's 1-99% range
